# HR Analytics – Predict Employee Attrition
## Phase 2: Data Preprocessing

**Objective:** Transform raw HR records into analysis-ready and model-ready datasets. Preprocessing is where many real-world projects succeed or fail — the goal is not to "clean until empty" but to make deliberate decisions that preserve business meaning while preparing data for EDA and machine learning.

**Inputs from Phase 1:**
- 1,470 employees, no missing values, no duplicates
- Target: `Attrition` (imbalanced ~16% Yes)
- Drop candidates: `EmployeeCount`, `StandardHours`, `Over18`, `EmployeeNumber`

---

## Setup & Load Raw Data

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

# Project paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "WA_Fn-UseC_-HR-Employee-Attrition.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Import reusable preprocessing functions from src/
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from preprocessing import (
    NOMINAL_COLS,
    ORDINAL_COLS,
    build_clean_dataset,
    build_ml_dataset,
    detect_outliers_iqr,
    drop_non_predictive_columns,
    encode_features_onehot,
    encode_target,
    load_raw_data,
    remove_duplicates,
    treat_missing_values,
    create_salary_band,
    create_experience_group,
)

df_raw = load_raw_data(DATA_PATH)
print(f"Raw shape: {df_raw.shape}")
df_raw.head(3)

---
## 1. Missing Value Treatment

**Why this step matters:** Missing HR data (salary blanks, incomplete satisfaction surveys) can bias attrition models if handled carelessly. Median imputation for numeric fields and mode for categoricals are conservative defaults that avoid distorting distributions.

**Decision for this dataset:** Phase 1 confirmed **zero missing values**. We still run the treatment function to keep the pipeline production-ready — if HRIS exports change tomorrow, the same code applies imputation automatically.

In [ ]:
missing_before = df_raw.isnull().sum().sum()
df = treat_missing_values(df_raw)
missing_after = df.isnull().sum().sum()

print(f"Missing values before treatment: {missing_before}")
print(f"Missing values after treatment:  {missing_after}")
print("\n→ No imputation applied — dataset was already complete.")

---
## 2. Duplicate Removal

**Why:** Duplicate employee records would inflate headcount and understate attrition rate. We check both full-row duplicates and duplicate `EmployeeNumber` values.

**Decision:** No duplicates exist. The function remains in the pipeline as a safeguard.

In [ ]:
rows_before = len(df)
df = remove_duplicates(df)
rows_after = len(df)

print(f"Rows before duplicate check: {rows_before}")
print(f"Rows after duplicate check:  {rows_after}")
print(f"Rows removed: {rows_before - rows_after}")

---
## 3. Remove Non-Predictive Columns

**Why:** Constant columns (`EmployeeCount`, `StandardHours`, `Over18`) carry zero variance — models cannot learn from them. `EmployeeNumber` is an identifier, not a behavioral feature; including it would cause overfitting to specific people.

**Decision:** Drop all four columns before EDA and modeling.

In [ ]:
print("Columns removed:", ["EmployeeCount", "StandardHours", "Over18", "EmployeeNumber"])
df = drop_non_predictive_columns(df)
print(f"Shape after column removal: {df.shape}")
df.columns.tolist()

---
## 4. Outlier Detection

**Why:** Extreme values in income or tenure may be data errors — or they may represent senior executives and long-tenured staff. Blind removal would delete valid business cases.

**Method:** Interquartile Range (IQR) rule — values below Q1 − 1.5×IQR or above Q3 + 1.5×IQR are flagged as outliers.

**Decision:** **Flag only, do not remove.** In HR analytics, a employee with 40 years at the company is unusual but real. We document outliers and retain all records.

In [ ]:
# Numeric fields where outliers are most likely to affect interpretation
OUTLIER_COLS = [
    "Age",
    "MonthlyIncome",
    "DistanceFromHome",
    "TotalWorkingYears",
    "YearsAtCompany",
    "YearsSinceLastPromotion",
]

outlier_flags = detect_outliers_iqr(df, OUTLIER_COLS)

outlier_summary = pd.DataFrame({
    "Outlier Count": [outlier_flags[c].sum() for c in outlier_flags.columns if c != "any_outlier"],
    "Outlier %": [
        round(outlier_flags[c].mean() * 100, 2)
        for c in outlier_flags.columns
        if c != "any_outlier"
    ],
}, index=[c.replace("_outlier", "") for c in outlier_flags.columns if c != "any_outlier"])

print(f"Employees flagged with at least one outlier: {outlier_flags['any_outlier'].sum()} "
      f"({outlier_flags['any_outlier'].mean()*100:.1f}%)")
outlier_summary

In [ ]:
# Boxplots for key variables — visual confirmation of spread and flagged extremes
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

for ax, col in zip(axes, OUTLIER_COLS):
    sns.boxplot(y=df[col], ax=ax, color="steelblue")
    ax.set_title(col)
    ax.set_ylabel("")

plt.suptitle("Outlier Detection — IQR Boxplots (points beyond whiskers = flagged outliers)", y=1.02)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "images" / "phase2_outlier_boxplots.png", dpi=120, bbox_inches="tight")
plt.show()

**Interpretation:**
- `MonthlyIncome`, `YearsAtCompany`, and `YearsSinceLastPromotion` show the most IQR flags (~7–8% each) — typically high earners or employees with long tenure/no recent promotion.
- `Age` and `DistanceFromHome` have no IQR outliers — distributions are compact.
- **HR implication:** Retention programs should not ignore senior, high-paid staff; their "outlier" status may correlate with flight risk when promotions stall.

---
## 5. Feature Engineering (Initial)

Raw numeric fields like `MonthlyIncome` and `TotalWorkingYears` are precise but hard to communicate to HR stakeholders. Banding creates segments that align with policy discussions (pay grades, career stages).

Phase 4 will add further engineered features (`Age_Group`, `Tenure_Category`, `Promotion_Category`). Here we create the two most impactful for EDA and Power BI: **Salary Band** and **Experience Group**.

### 5.1 Salary Band

**Why:** HR teams think in compensation tiers, not exact dollar amounts. Quartile-based bands (`Low`, `Medium`, `High`, `Very High`) reflect the actual income distribution in *this* company rather than arbitrary industry cutoffs.

**Method:** `pd.qcut` on `MonthlyIncome` into 4 equal-frequency groups.

In [ ]:
df = create_salary_band(df)

# Show income ranges behind each band — helps validate business meaning
salary_band_profile = df.groupby("Salary_Band", observed=True)["MonthlyIncome"].agg(
    ["min", "max", "mean", "count"]
).round(0)
salary_band_profile

### 5.2 Experience Group

**Why:** Total working years captures career maturity and external market mobility. Early-career employees often leave for growth; veterans may leave due to stagnation — different interventions apply.

**Bins:** 0–5 (Early), 6–10 (Mid), 11–20 (Experienced), 20+ (Veteran).

In [ ]:
df = create_experience_group(df)

exp_profile = df.groupby("Experience_Group", observed=True).agg(
    employees=("Age", "count"),
    avg_age=("Age", "mean"),
    avg_income=("MonthlyIncome", "mean"),
).round(1)
exp_profile

---
## 6. Target Encoding

**Why:** Machine learning models require numeric targets. We create `Attrition_Flag`: 1 = left, 0 = stayed. The original `Attrition` text column is kept for EDA readability.

In [ ]:
df = encode_target(df)
print(df[["Attrition", "Attrition_Flag"]].drop_duplicates().sort_values("Attrition_Flag"))
print(f"\nAttrition rate (flag mean): {df['Attrition_Flag'].mean():.2%}")

---
## 7. Categorical Variable Encoding

Different variable types need different treatment:

| Type | Examples | Strategy | Reason |
|------|----------|----------|--------|
| **Nominal** | Department, JobRole, Gender | One-hot encoding (ML) / keep raw (EDA) | No natural order between categories |
| **Ordinal** | JobSatisfaction (1–4), Education (1–5) | Keep as integers | Order is meaningful; trees handle this well |
| **Binary** | OverTime, Attrition | Yes/No → 0/1 | Already two levels |

**Why `drop_first=True`:** Avoids the dummy variable trap (perfect multicollinearity) — important for Logistic Regression in Phase 5.

In [ ]:
print("Nominal columns (one-hot encoded for ML):")
print(NOMINAL_COLS)
print("\nOrdinal columns (retained as integers):")
print(ORDINAL_COLS)

# Build ML-ready dataset: drop text target, one-hot encode nominals
df_ml = build_ml_dataset(df)

print(f"\nEDA-ready shape (human-readable categories): {df.shape}")
print(f"ML-ready shape (one-hot encoded):            {df_ml.shape}")
print(f"\nNew one-hot columns sample:")
[c for c in df_ml.columns if "_" in c and c not in df.columns][:8]

In [ ]:
# Preview encoded feature matrix
df_ml.head(3)

---
## 8. Feature Scaling

**Why:** Algorithms like Logistic Regression are sensitive to feature magnitude — `MonthlyIncome` (thousands) would dominate `JobSatisfaction` (1–4) without scaling. Tree-based models (Decision Tree, Random Forest) are **scale-invariant** and do not require normalization.

**Decision:**
- **Do NOT scale the saved EDA dataset** — bands and raw values are easier to interpret.
- **Apply `StandardScaler` at model training time (Phase 5)** on the training split only to prevent data leakage.
- Below we demonstrate scaling on numeric columns for documentation purposes.

In [ ]:
# Numeric columns to scale (exclude target flag and one-hot dummies)
SCALE_COLS = [
    c for c in df_ml.columns
    if df_ml[c].dtype in [np.int64, np.float64]
    and c not in ["Attrition_Flag"]
    and not c.startswith(("BusinessTravel_", "Department_", "EducationField_",
                          "Gender_", "JobRole_", "MaritalStatus_", "OverTime_"))
]

scaler = StandardScaler()
df_scaled_preview = df_ml.copy()
df_scaled_preview[SCALE_COLS] = scaler.fit_transform(df_ml[SCALE_COLS])

print(f"Columns scaled (preview only): {len(SCALE_COLS)}")
print("\nBefore vs after scaling — MonthlyIncome:")
print(f"  Original mean: {df_ml['MonthlyIncome'].mean():.2f}, std: {df_ml['MonthlyIncome'].std():.2f}")
print(f"  Scaled mean:   {df_scaled_preview['MonthlyIncome'].mean():.4f}, std: {df_scaled_preview['MonthlyIncome'].std():.4f}")

**Note:** The scaler is intentionally **not** persisted here. Phase 5 will fit `StandardScaler` inside a sklearn `Pipeline` on training data only — the correct industry practice.

---
## 9. Final Dataset Assembly & Export

We save two artifacts:
1. **`hr_cleaned.csv`** — For EDA (Phase 3), Power BI (Phase 7), and business reporting. Readable categories + engineered features.
2. **`hr_ml_ready.csv`** — One-hot encoded features for modeling (Phase 5).

In [ ]:
# Attach outlier flag for optional filtering in analysis (not used in modeling by default)
df["Any_Outlier_Flag"] = outlier_flags["any_outlier"].astype(int)

CLEAN_PATH = PROCESSED_DIR / "hr_cleaned.csv"
ML_PATH = PROCESSED_DIR / "hr_ml_ready.csv"

df.to_csv(CLEAN_PATH, index=False)
df_ml.to_csv(ML_PATH, index=False)

print(f"Saved EDA-ready dataset:  {CLEAN_PATH}")
print(f"Saved ML-ready dataset:   {ML_PATH}")
print(f"\nFinal EDA dataset shape: {df.shape}")
print(f"Final ML dataset shape:  {df_ml.shape}")

In [ ]:
# Preprocessing audit trail — documents every decision for reproducibility
audit = pd.DataFrame([
    {"Step": "Missing values", "Action": "Median (numeric) / Mode (categorical)", "Result": "None found — no imputation"},
    {"Step": "Duplicates", "Action": "Drop exact + EmployeeNumber dupes", "Result": "0 rows removed"},
    {"Step": "Column removal", "Action": "Drop 4 non-predictive cols", "Result": "35 → 31 columns, then +3 engineered/target"},
    {"Step": "Outliers", "Action": "IQR flag only", "Result": "Retained all rows"},
    {"Step": "Feature engineering", "Action": "Salary_Band, Experience_Group", "Result": "2 new features"},
    {"Step": "Target encoding", "Action": "Attrition_Flag (Yes=1)", "Result": "16.12% positive class"},
    {"Step": "Categorical encoding", "Action": "One-hot (nominal), keep ordinal ints", "Result": f"{df_ml.shape[1]} ML columns"},
    {"Step": "Scaling", "Action": "Deferred to Phase 5 Pipeline", "Result": "Not applied to saved files"},
])
audit

---

## Phase 2 Complete — Preprocessing Summary

| Deliverable | Location | Purpose |
|-------------|----------|--------|
| Reusable preprocessing module | `src/preprocessing.py` | Consistent transforms across notebooks |
| EDA-ready dataset | `data/processed/hr_cleaned.csv` | Phase 3 EDA, Phase 7 Power BI |
| ML-ready dataset | `data/processed/hr_ml_ready.csv` | Phase 5 modeling |
| Outlier visualization | `images/phase2_outlier_boxplots.png` | Documentation |

### Key Decisions Recap

1. **No rows dropped** — data quality was already high; outlier flags preserved for transparency.
2. **Salary & experience bands** added for stakeholder-friendly analysis.
3. **Two dataset versions** — readable for humans, encoded for algorithms.
4. **Scaling deferred** to training pipeline — prevents leakage and respects model-specific needs.

**Next step (Phase 3):** Exploratory Data Analysis with visualizations and business insights for every chart.

*Awaiting confirmation to proceed to Phase 3 – Exploratory Data Analysis.*